# Construction du dataset d'entrainement

In [1]:
import pandas as pd
import numpy as np

from scipy.spatial import cKDTree
from tqdm import tqdm

tqdm.pandas()

In [2]:
# Population Maroc
pop1 = pd.read_csv("population_by_surface/mor_pop.csv")

# Population Sahara
pop2 = pd.read_csv("population_by_surface/mor_sahara_pop.csv")

# Age WorldPop
age = pd.read_csv("age_by_surface/morocco_age_csv_version.csv")

In [3]:
population = pd.concat([pop1, pop2], ignore_index=True)

population = population.rename(columns={
    "X":"longitude",
    "Y":"latitude",
    "Z":"population"
})

population = population[["longitude","latitude","population"]]

population.head()

,longitude,latitude,population
0,-5.422083,35.920417,126.284340
1,-5.413750,35.920417,124.144516
2,-5.405417,35.920417,204.392166
3,-5.397083,35.920417,305.608734
4,-5.480417,35.912083,116.053368


In [4]:
total_cols = [c for c in age.columns if c.startswith("mar_t_")]

age["population_totale"] = age[total_cols].sum(axis=1)

In [5]:
age["population_femmes"] = age["mar_T_F_2026_CN_100m_R2025A_v1"]

age["population_hommes"] = age["mar_T_M_2026_CN_100m_R2025A_v1"]

In [6]:
enfants = [
    "mar_t_00_2026_CN_100m_R2025A_v1",
    "mar_t_01_2026_CN_100m_R2025A_v1",
    "mar_t_05_2026_CN_100m_R2025A_v1",
    "mar_t_10_2026_CN_100m_R2025A_v1"
]

age["enfants"] = age[enfants].sum(axis=1)

In [7]:
jeunes = [
    "mar_t_15_2026_CN_100m_R2025A_v1",
    "mar_t_20_2026_CN_100m_R2025A_v1"
]

age["jeunes"] = age[jeunes].sum(axis=1)

In [8]:
actifs = [
    "mar_t_25_2026_CN_100m_R2025A_v1",
    "mar_t_30_2026_CN_100m_R2025A_v1",
    "mar_t_35_2026_CN_100m_R2025A_v1",
    "mar_t_40_2026_CN_100m_R2025A_v1",
    "mar_t_45_2026_CN_100m_R2025A_v1",
    "mar_t_50_2026_CN_100m_R2025A_v1",
    "mar_t_55_2026_CN_100m_R2025A_v1"
]

age["actifs"] = age[actifs].sum(axis=1)

In [9]:
seniors = [
    "mar_t_60_2026_CN_100m_R2025A_v1",
    "mar_t_65_2026_CN_100m_R2025A_v1",
    "mar_t_70_2026_CN_100m_R2025A_v1",
    "mar_t_75_2026_CN_100m_R2025A_v1",
    "mar_t_80_2026_CN_100m_R2025A_v1",
    "mar_t_85_2026_CN_100m_R2025A_v1",
    "mar_t_90_2026_CN_100m_R2025A_v1"
]

age["seniors"] = age[seniors].sum(axis=1)

In [10]:
age["ratio_enfants"] = age["enfants"] / age["population_totale"]

age["ratio_jeunes"] = age["jeunes"] / age["population_totale"]

age["ratio_actifs"] = age["actifs"] / age["population_totale"]

age["ratio_seniors"] = age["seniors"] / age["population_totale"]

age["ratio_femmes"] = age["population_femmes"] / age["population_totale"]

In [11]:
age = age[
    [
        "lon",
        "lat",
        "ratio_enfants",
        "ratio_jeunes",
        "ratio_actifs",
        "ratio_seniors",
        "ratio_femmes"
    ]
]

age.head()

,lon,lat,ratio_enfants,ratio_jeunes,ratio_actifs,ratio_seniors,ratio_femmes
0,-5.402084,35.920418,0.248375,0.165161,0.469708,0.116756,0.483432
1,-5.401251,35.920418,0.248374,0.165162,0.469707,0.116757,0.483431
2,-5.404584,35.919582,0.248375,0.165159,0.469706,0.116760,0.483431
3,-5.403751,35.919582,0.248374,0.165161,0.469709,0.116755,0.483432
4,-5.402917,35.919582,0.248375,0.165162,0.469707,0.116755,0.483431


In [12]:
tree = cKDTree(age[["lon","lat"]].values)

In [13]:
RAYON = 0.0065

In [14]:
indices = tree.query_ball_point(
    population[["longitude","latitude"]].values,
    r=RAYON
)

In [15]:
ratio_cols = [
    "ratio_enfants",
    "ratio_jeunes",
    "ratio_actifs",
    "ratio_seniors",
    "ratio_femmes"
]

for col in ratio_cols:
    population[col] = np.nan

In [16]:
for i, voisins in tqdm(enumerate(indices), total=len(indices)):

    if len(voisins)==0:
        continue

    population.loc[i, ratio_cols] = age.iloc[voisins][ratio_cols].mean().values

100%|██████████| 912151/912151 [02:04<00:00, 7318.52it/s]  


In [17]:
population.to_csv(
    "training_dataset.csv",
    index=False
)

print("Dataset créé :", population.shape)

Dataset créé : (912151, 8)


In [18]:
population.head()

,longitude,latitude,population,ratio_enfants,ratio_jeunes,ratio_actifs,ratio_seniors,ratio_femmes
0,-5.422083,35.920417,126.284340,NaN,NaN,NaN,NaN,NaN
1,-5.413750,35.920417,124.144516,NaN,NaN,NaN,NaN,NaN
2,-5.405417,35.920417,204.392166,0.248375,0.165161,0.469708,0.116756,0.483432
3,-5.397083,35.920417,305.608734,0.248375,0.165161,0.469708,0.116756,0.483432
4,-5.480417,35.912083,116.053368,0.256679,0.183634,0.451455,0.108232,0.476268
